# L14 · 特征值与 SVD — Ax=λx 的几何意义，SVD 作为万能分解工具

**目标**：建立特征向量（变换中方向不变的轴）、PCA（找主要变化方向降维）、SVD（万能矩阵分解）。

**为什么对 Aurora 重要**：SVD 是音频去噪（截断低秩 STFT 矩阵）、embedding 降维可视化（PCA）和 LoRA 微调（低秩权重适配）的共同数学基础。

## 本节内容

本节从方阵特征值问题 `A v = λ v` 出发，用 `np.linalg.eig(A)` 验证特征向量「方向不变」的性质。随后推进到 SVD 分解 `A = U · S · Vt`（旋转→奇异值缩放→旋转），并实现 `low_rank_approx(A, k)` 截断前 k 个奇异值重建矩阵——Aurora 音频去噪和 LoRA 微调均基于此操作。

## 1. 特征向量：`A v = λ v`（A 作用在 v 上只改变长度、不改变方向）

## 符号入口：本节关键 shape

本节的关键 shape 变化：输入矩阵 A 是 `(m, n)`，特征向量 v 是 `(n,)`，奇异值向量 S 是 `(min(m,n),)`，U 是 `(m, m)`，Vt 是 `(n, n)`。特征值问题 `Av = λv` 要求 A 是方阵；SVD 对任意矩阵都成立。

## 1.1 特征向量和普通向量的差别

同一个矩阵作用到不同方向上，结果可能会转弯，也可能只变长。特征向量就是“不转弯”的方向。


In [ ]:
import numpy as np

A = np.array([[2.0, 0.0], [0.0, 3.0]])
eigen_v = np.array([1.0, 0.0])
ordinary_v = np.array([1.0, 1.0])

print('A @ eigen_v   =', A @ eigen_v, ' 方向不变')
print('A @ ordinary_v=', A @ ordinary_v, ' 方向改变')


In [ ]:
import numpy as np
A = np.array([[2.0,0.0],[0.0,3.0]])
vals, vecs = np.linalg.eig(A)
print('特征值 λ:', vals)          # [2. 3.]
print('特征向量(按列):\n', vecs)  # x轴、y轴

## 动手观察

对角矩阵的特征向量正好是坐标轴方向：检查 `A @ vecs[:, i]` 与 `vals[i] * vecs[:, i]` 的数值差是否小于 `1e-10`，再换一个非特征方向的向量验证方向确实偏转。

In [ ]:
import numpy as np

v = np.array([3.0, 4.0])
A = np.array([[2.0, 0.0],
              [0.0, 0.5]])

print('v =', v, 'shape =', v.shape)
print('A =')
print(A)
print('A shape =', A.shape)
print('A @ v =', A @ v)
print('向量长度 ||v|| =', np.linalg.norm(v))


## 代码实验：遍历几个向量，观察矩阵如何改变它们

对比普通向量和特征向量经过同一矩阵后的方向变化，直接观察 `A @ v` 和 `λ * v` 的数值差。

In [ ]:
import numpy as np

A = np.array([[2.0, 1.0],
              [0.0, 1.0]])
vectors = [np.array([1.0, 0.0]), np.array([0.0, 1.0]), np.array([1.0, 1.0]), np.array([2.0, -1.0])]

print('A =')
print(A)
for v in vectors:
    out = A @ v
    print(f'v={v} -> A@v={out}')


## 2. PCA：先找数据变化最大的方向

PCA 可以看成“找一组新坐标轴”：第一根轴对准数据变化最大的方向。SVD 会在后面给出更通用的分解工具。


In [ ]:
import numpy as np, matplotlib.pyplot as plt

X = np.array([
    [0.0, 0.2], [1.0, 1.1], [2.0, 1.9], [3.0, 3.2], [4.0, 3.9]
])
Xc = X - X.mean(axis=0)
U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
principal = Vt[0]

plt.figure(figsize=(4.5, 4.5))
plt.scatter(X[:,0], X[:,1], label='data')
center = X.mean(axis=0)
line = np.vstack([center - principal*2.5, center + principal*2.5])
plt.plot(line[:,0], line[:,1], label='first principal direction')
plt.axis('equal'); plt.grid(True, alpha=.25); plt.legend(); plt.show()
print('principal direction =', np.round(principal, 3))


## 2. SVD：任何矩阵都能拆成 `A = U · S · Vᵀ`（S 是奇异值,衡量每个方向的重要性）

In [ ]:
A = np.array([[3.0,1.0,1.0],[1.0,3.0,1.0],[1.0,1.0,3.0]])
U, S, Vt = np.linalg.svd(A)
print('奇异值 S:', np.round(S, 3))
recon = U @ np.diag(S) @ Vt
print('U·S·Vᵀ 还原 A?', np.allclose(recon, A))

## 2.1 SVD：把矩阵拆成方向和强度

`A = U · S · Vᵀ` 可以读成三步：

1. `Vᵀ`：换到一组输入方向。
2. `S`：沿这些方向拉伸或压缩。
3. `U`：转到输出方向。

低秩近似就是只保留最大的几个拉伸强度。


## 3. ✏️ 你的任务：低秩近似（只保留最大的 k 个奇异值）

这正是 LoRA / 推荐里"压缩矩阵"的核心。
**提示**：`U[:, :k] @ np.diag(S[:k]) @ Vt[:k, :]`。

### 写 `low_rank_approx` 前明确三件事

- 输入：矩阵 `A`（shape `(m, n)`）和保留的秩 `k`
- 关键步骤：对 `A` 做 SVD，取前 `k` 列/行：`U[:, :k] @ np.diag(S[:k]) @ Vt[:k, :]`
- 返回：shape 与 `A` 相同的秩-k 近似矩阵，`k` 越小，误差越大

In [ ]:
def low_rank_approx(A, k):
    U, S, Vt = np.linalg.svd(A)
    # ✏️ TODO: 用前 k 个奇异值重建 A 的近似
    return None  # ← 改这里


In [ ]:
A = np.array([[3.0,1.0,1.0],[1.0,3.0,1.0],[1.0,1.0,3.0]])
A1 = low_rank_approx(A, 1)   # 只用最重要的 1 个方向
A3 = low_rank_approx(A, 3)   # 用全部 -> 完全还原
print('rank-1 近似:\n', np.round(A1, 2))
print('rank-3 应= A:', np.allclose(A3, A))
assert np.allclose(A3, A), 'k=满秩 应完全还原'
print('\n✅ 通过：你做出了 LoRA/推荐 背后的核心操作。')

**🔗 Aurora 连接**：Month 4 推荐(矩阵分解)、Month 5 LoRA 微调(低秩适配)、embedding 降维可视化——全是今天这套 SVD 思想。

**🎉 前导线代课完成**：你已握住 Aurora 工程里最常用的线代工具。下一块数学(微积分/概率)等进 Month 2 深度学习前再补。

## 🎨 图示：矩阵 = 秩1矩阵之和 (SVD 的核心画面)

In [ ]:
from laviz import style, mat_times_mat_rank1
style()
mat_times_mat_rank1([[1,2],[3,4]],[[5,6],[7,8]]);

In [ ]:
A = np.array([[2.0, 1.0], [0.0, 1.0]])
probes = np.array([[1,0], [0,1], [1,1], [-1,2]], dtype=float)
print('矩阵 A 会怎样移动这些向量？')
for v in probes:
    out = A @ v
    print(f'{v} -> {out} | 长度 {np.linalg.norm(v):.2f} -> {np.linalg.norm(out):.2f}')


## 参数实验：只改一个旋钮

把 `k` 从 1 改到矩阵满秩，观察低秩近似误差 `||A - Ak||` 如何随 k 减小。再把矩阵 A 换成秩为 1 的矩阵，确认 k=1 时近似误差为零。

In [ ]:
A = np.array([[2.0, 1.0], [0.0, 1.0]])
probes = np.array([[1,0], [0,1], [1,1], [-1,2]], dtype=float)
print('矩阵 A 会怎样移动这些向量？')
for v in probes:
    out = A @ v
    print(f'{v} -> {out} | 长度 {np.linalg.norm(v):.2f} -> {np.linalg.norm(out):.2f}')


## 本课收束

`np.linalg.eig(A)` 返回特征值数组 `vals` 和特征向量矩阵 `vecs`，可逐列验证 `A @ vecs[:, i] ≈ vals[i] * vecs[:, i]`。`np.linalg.svd(A, full_matrices=True)` 返回 `U`、`S`、`Vt`，用 `U[:, :k] @ np.diag(S[:k]) @ Vt[:k, :]` 即得 `low_rank_approx(A, k)` 的核心运算。Aurora 音频去噪管线在 STFT 频谱矩阵上截断最小奇异值分量，抑制低能量噪声背景。LoRA 微调把预训练权重矩阵表示为两个低秩因子之积，是同一套截断 SVD——下一节（p7 线性方程组）会用特征值判断系数矩阵是否可逆。

In [ ]:
# 小检查：先看 shape，再做运算
import numpy as np

a = np.array([1.0, 2.0])
b = np.array([3.0, 4.0])
A = np.array([[1.0, 2.0], [0.0, 1.0]])

print('a.shape =', a.shape)
print('b.shape =', b.shape)
print('A.shape =', A.shape)
print('a + b =', a + b)
print('A @ a =', A @ a)
print('a dot b =', float(a @ b))
